In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from scipy.stats import median_abs_deviation
import warnings
import random

warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# 1. CONSTANTS & SETUP
# ============================================================================
SEED = 99
random.seed(SEED)
np.random.seed(SEED)

WINDOW_SIZE_SPREAD = 100   # lookback for IQR winsorization and MAD scale
WINDOW_SIZE_LOCATION = 100 # lookback for rolling median (centering baseline)

# Number of z-score observations used to compute a single LMAD value.
# At 10 Hz WiFi CSI sample rate, 53 samples ~ 5.3 seconds of signal.
WINDOW_SIZE_FEATURE = 53

# Non-overlapping segment size for aggregation. Each segment produces one
# evaluation sample (median of LMAD values within the segment). Using the
# feature window size ensures roughly independent segments.
SEGMENT_SIZE = WINDOW_SIZE_FEATURE

N_PCA_COMPONENTS = 1

# Search range for differencing order. Extended to n=8 to observe the full
# rise-peak-fall curve. For 10 Hz CSI, orders beyond ~6 are noise-dominated,
# so n=8 is sufficient to confirm convergence/divergence.
MAX_N_DIFF = 8

TRAIN_SUBJECTS = ['matthew', 'kenny', 'collin']
TEST_SUBJECTS = ['ivan', 'abel']

In [3]:
# ============================================================================
# 2. HELPER FUNCTIONS (PCA & FEATURE ENGINEERING)
# ============================================================================
def calculate_mad_scale(x):
    x_clean = x[~np.isnan(x)]
    if len(x_clean) < 1: return np.nan
    return median_abs_deviation(x_clean, scale='normal', nan_policy='omit')

def calculate_log_mean_abs_diff(x, n_diff):
    x_clean = x[~np.isnan(x)]
    if len(x_clean) < n_diff + 1: return np.nan
    d = np.diff(x_clean, n=n_diff)
    mean_abs_d = np.mean(np.abs(d))
    # FIX: return NaN instead of -inf for flat windows.
    # -inf poisons np.median/nanmedian and roc_auc_score downstream.
    if mean_abs_d == 0: return np.nan
    return np.log(mean_abs_d)

def apply_pca_separate(df, scidx_mag_cols, scidx_phase_cols, train_subjects):
    df = df.copy()
    if scidx_mag_cols: df['PCA_Mag'] = np.nan
    if scidx_phase_cols: df['PCA_Phase'] = np.nan

    for env in df['environment'].unique():
        mask_env = df['environment'] == env
        # FIX: Fit PCA on ALL activities from train subjects, not standing-only.
        # Standing-only fit captures idle-state variance (noise/drift), not the
        # directions that separate activities. Using all activities lets PCA find
        # components that span the full signal space.
        mask_fit = mask_env & df['subject'].isin(train_subjects)

        if not mask_fit.any(): continue

        #Magnitude PCA
        if scidx_mag_cols:
            mag_fit = df.loc[mask_fit, scidx_mag_cols].values
        
            if np.isnan(mag_fit).all():
                continue
                
            col_means = np.nanmean(mag_fit, axis=0)
            col_means = np.where(np.isnan(col_means), 0, col_means)
            inds = np.where(np.isnan(mag_fit))
            mag_fit[inds] = np.take(col_means, inds[1])
            
            pca_mag = PCA(n_components=N_PCA_COMPONENTS, random_state=SEED)
            pca_mag.fit(mag_fit)
    
            mag_all = df.loc[mask_env, scidx_mag_cols].values
            inds_all = np.where(np.isnan(mag_all))
            mag_all[inds_all] = np.take(col_means, inds_all[1])
            df.loc[mask_env, 'PCA_Mag'] = pca_mag.transform(mag_all)[:, 0]

        #Phase PCA
        if scidx_phase_cols:
            phase_fit = df.loc[mask_fit, scidx_phase_cols].values
        
            if np.isnan(phase_fit).all():
                continue
                
            col_means = np.nanmean(phase_fit, axis=0)
            col_means = np.where(np.isnan(col_means), 0, col_means)
            inds = np.where(np.isnan(phase_fit))
            phase_fit[inds] = np.take(col_means, inds[1])
            
            pca_phase = PCA(n_components=N_PCA_COMPONENTS, random_state=SEED)
            pca_phase.fit(phase_fit)

            phase_all = df.loc[mask_env, scidx_phase_cols].values
            inds_all = np.where(np.isnan(phase_all))
            phase_all[inds_all] = np.take(col_means, inds_all[1])
            df.loc[mask_env, 'PCA_Phase'] = pca_phase.transform(phase_all)[:, 0]

    return df

def extract_lmad_feature(df_in, base_col, n_diff):
    """
    Extract LMAD (Log Mean Absolute Difference) feature per sample.
    
    Pipeline per session:
      1. Causal IQR winsorization (clip outliers using past window)
      2. Rolling z-score standardization (median + MAD from past window)
      3. Rolling LMAD on z-scores (log of mean absolute n-th difference)
    
    NOTE: The inner loops are O(n * window_size) per session. For large-scale
    use, consider vectorizing with pandas.rolling or numba JIT compilation.
    """
    df = df_in.copy()
    feat_col_name = f"{base_col}_ord{n_diff}"
    value_col = df[base_col].values
    session_ids = df['session_id'].values
    
    n = len(df)
    roll_winsorized_value = np.full(n, np.nan)
    roll_median_winsor = np.full(n, np.nan)
    roll_mad_scale_winsor = np.full(n, np.nan)
    z_scores = np.full(n, np.nan)
    feat_values = np.full(n, np.nan)
    
    for session_id in np.unique(session_ids):
        indices = np.where(session_ids == session_id)[0]
        vals = value_col[indices]
        
        # 1. Winsorization (causal window = WINDOW_SIZE_SPREAD)
        # Clips current value against IQR bounds computed from past observations.
        for i in range(len(indices)):
            global_idx = indices[i]
            if i == 0:
                roll_winsorized_value[global_idx] = vals[i]
                continue
            start_idx = max(0, i - WINDOW_SIZE_SPREAD)
            window = vals[start_idx:i]
            if len(window) >= 4:
                q1, q3 = np.nanquantile(window, [0.25, 0.75])
                iqr = q3 - q1
                roll_winsorized_value[global_idx] = np.clip(vals[i], q1 - 1.5*iqr, q3 + 1.5*iqr)
            else:
                roll_winsorized_value[global_idx] = vals[i]

    for session_id in np.unique(session_ids):
        indices = np.where(session_ids == session_id)[0]
        winsor_vals = roll_winsorized_value[indices]
        
        # 2. Standardization: z = (winsorized - rolling_median) / rolling_MAD
        for i in range(len(indices)):
            if i == 0: continue
            global_idx = indices[i]
            
            start_loc = max(0, i - WINDOW_SIZE_LOCATION)
            win_loc = winsor_vals[start_loc:i]
            if len(win_loc) >= 1: roll_median_winsor[global_idx] = np.nanmedian(win_loc)
            
            start_spr = max(0, i - WINDOW_SIZE_SPREAD)
            win_spr = winsor_vals[start_spr:i]
            if len(win_spr) >= 1: roll_mad_scale_winsor[global_idx] = calculate_mad_scale(win_spr)
                
            mad = roll_mad_scale_winsor[global_idx]
            if not np.isnan(mad) and mad != 0:
                z_scores[global_idx] = (winsor_vals[i] - roll_median_winsor[global_idx]) / mad
            elif mad == 0:
                z_scores[global_idx] = 0
            
            # 3. LMAD: log(mean(|diff^n(z)|)) over feature window
            if i >= WINDOW_SIZE_FEATURE - 1:
                win_z_start = i - WINDOW_SIZE_FEATURE + 1
                win_z = z_scores[indices[win_z_start : i+1]]
                if len(win_z) == WINDOW_SIZE_FEATURE:
                    feat_values[global_idx] = calculate_log_mean_abs_diff(win_z, n_diff)

    df[feat_col_name] = feat_values
    return df

In [ ]:
# ============================================================================
# 3. MAIN EXECUTION PIPELINE
# ============================================================================
print("1. Loading & Filtering Data...")
df_raw = pd.read_csv("/kaggle/input/datasets/purpleginseng/bfmsingleapsensing/bfm_data.csv")

scidx_mag_cols = [c for c in df_raw.columns if 'SCIDX_' in c and '_Ratio_Mag' in c]
scidx_phase_cols = [c for c in df_raw.columns if 'SCIDX_' in c and '_Ratio_Phase' in c]

# Magnitude Filter (> 5.0)
df_raw['Magnitude'] = df_raw[scidx_mag_cols].mean(axis=1)
df_clean = df_raw[df_raw['Magnitude'] > 5.0].copy()

#Sanity Check after filtering
print(f"Length of df_raw: {len(df_raw)}")
print(f"Length of df_clean: {len(df_clean)}")

all_sessions_df_raw = df_raw['session_id'].unique()
all_sessions_df_clean = df_clean['session_id'].unique()

print("We have all the sessions") if len(all_sessions_df_clean) == len(all_sessions_df_raw) else print("There is at least one missing session.")
session_counts = df_clean.groupby(['environment', 'session_id']).size().reset_index(name='row_count')
collapse = session_counts.sort_values('row_count').head(10)
print("10 sessions with the least rows")
print(collapse)

threshold = max(WINDOW_SIZE_SPREAD, WINDOW_SIZE_LOCATION, WINDOW_SIZE_FEATURE)
broken_sessions = session_counts[session_counts["row_count"] < threshold]
print(f"\nNumber of broken sessions (lengths < {threshold}): {len(broken_sessions)}")

the_problem_session = df_clean[df_clean["session_id"] == "20251016_183114"]
print(the_problem_session["activity"].unique())

df_clean = df_clean[~df_clean['session_id'].isin(broken_sessions['session_id'])]

# ============================================================================
# 4. LEAVE-ONE-SUBJECT-OUT CV FOR n_diff SELECTION
# ============================================================================
envs = ['nofoil', 'foil', 'open']
margin_data = []
n_diff_results = []  # collect (n, auc, margin) for smarter selection

def segment_aggregate(df_feat, feat_cols):
    """
    Split each session into non-overlapping segments of SEGMENT_SIZE rows,
    then take the median feature value per segment. This yields many more
    evaluation samples than 1-per-session, and captures within-session
    variability (e.g., early vs late walking).
    """
    df_feat = df_feat.copy()
    df_feat['segment_id'] = df_feat.groupby('session_id').cumcount() // SEGMENT_SIZE

    agg_dict = {col: 'median' for col in feat_cols}
    seg_agg = df_feat.groupby(
        ['session_id', 'segment_id', 'environment', 'subject', 'activity']
    ).agg(agg_dict).reset_index()

    return seg_agg

print(f"\n2. Evaluating n_diff (1 to {MAX_N_DIFF}) using LOSO-CV on Training Subjects...")
print(f"   (Segment size = {SEGMENT_SIZE} samples per evaluation unit)")
ALL_SUBJECTS = TRAIN_SUBJECTS

for current_n in range(1, MAX_N_DIFF + 1):
    print(f"\n   -> Order n={current_n}")
    fold_auc_scores = []
    fold_margin_scores = []

    mag_col = f'PCA_Mag_ord{current_n}'
    phase_col = f'PCA_Phase_ord{current_n}'

    for held_out in ALL_SUBJECTS:
        fold_train = [s for s in ALL_SUBJECTS if s != held_out]
        fold_test = [held_out]

        df_fold = apply_pca_separate(df_clean, scidx_mag_cols, scidx_phase_cols, fold_train)

        df_held = df_fold[df_fold['subject'].isin(fold_test)].copy()
        df_held = extract_lmad_feature(df_held, 'PCA_Mag', current_n)
        df_held = extract_lmad_feature(df_held, 'PCA_Phase', current_n)

        session_agg = segment_aggregate(df_held, [mag_col, phase_col])

        for env in envs:
            env_data = session_agg[session_agg['environment'] == env]
            if env_data.empty:
                continue

            for modality in ['PCA_Mag', 'PCA_Phase']:
                feat_col = f'{modality}_ord{current_n}'

                valid = env_data[feat_col].notna()
                env_valid = env_data[valid]
                
                y_true = (env_valid['activity'] == 'walking').astype(int)
                X_val = env_valid[feat_col]

                if len(X_val) < 2 or y_true.nunique() < 2:
                    continue

                auc_val = roc_auc_score(y_true, X_val)
                auc_val = max(auc_val, 1 - auc_val)
                fold_auc_scores.append(auc_val)

                walk_vals = X_val[y_true == 1]
                stand_vals = X_val[y_true == 0]

                med_walk, med_stand = np.median(walk_vals), np.median(stand_vals)
                mad_walk = median_abs_deviation(walk_vals)
                mad_stand = median_abs_deviation(stand_vals)

                denominator = mad_walk + mad_stand
                if denominator == 0:
                    denominator = 1e-9

                margin = abs(med_walk - med_stand) / denominator
                fold_margin_scores.append(margin)

                margin_data.append({
                    'n_diff': current_n,
                    'Environment': env.capitalize(),
                    'Modality': modality,
                    'Fold': held_out,
                    'AUC': auc_val,
                    'Margin': margin,
                    'n_segments': len(X_val)
                })

                print(f"      Fold={held_out:<8} | Env: {env:<8} | {modality:<9} | AUC: {auc_val:.4f} | Margin: {margin:.4f} | n_seg={len(X_val)}")

    avg_auc = np.mean(fold_auc_scores) if fold_auc_scores else 0
    avg_margin = np.mean(fold_margin_scores) if fold_margin_scores else 0
    n_diff_results.append({'n': current_n, 'auc': avg_auc, 'margin': avg_margin})
    print(f"      >> Mean AUC: {avg_auc:.4f} | Mean Margin: {avg_margin:.4f}")

# ============================================================================
# 4b. SMART SELECTION: AUC-tolerance band, then margin, then simplicity
# ============================================================================
# When AUC is near-saturated (all > 0.95, differences < 1%), tiny AUC gains
# are within noise. Use a tolerance band: any n within AUC_TOLERANCE of the
# best AUC is considered "tied", then pick by highest margin (better class
# separation), then by lowest n (simpler = more generalizable).
AUC_TOLERANCE = 0.01  # 1% tolerance — differences smaller than this are noise

results_df = pd.DataFrame(n_diff_results)
max_auc = results_df['auc'].max()

# All candidates within tolerance of the best AUC
candidates = results_df[results_df['auc'] >= max_auc - AUC_TOLERANCE].copy()
# Among those, pick highest margin; break ties with lowest n
candidates = candidates.sort_values(['margin', 'n'], ascending=[False, True])
best_row = candidates.iloc[0]
best_n = int(best_row['n'])
best_avg_auc = best_row['auc']
best_avg_margin = best_row['margin']

print(f"\n--- Selection Logic ---")
print(f"   Max AUC across all n: {max_auc:.4f}")
print(f"   Tolerance band: AUC >= {max_auc - AUC_TOLERANCE:.4f}")
print(f"   Candidates in band: n = {sorted(candidates['n'].tolist())}")
print(f"   Selected by highest margin (+ lowest n tiebreak):")
print(f"\n[RESULT] Best differencing order: n={best_n} (Mean AUC={best_avg_auc:.4f}, Mean Margin={best_avg_margin:.4f})")

# Warn if best_n is at boundary
if best_n == MAX_N_DIFF:
    print(f"   [WARNING] n={best_n} is at the search boundary! Consider increasing MAX_N_DIFF.")

# ============================================================================
# 5. FINAL REFIT: PCA on full training set, evaluate on TEST subjects
# ============================================================================
print(f"\n3. Final evaluation on held-out TEST subjects {TEST_SUBJECTS}...")
df_pca = apply_pca_separate(df_clean, scidx_mag_cols, scidx_phase_cols, TRAIN_SUBJECTS)

df_test = df_pca[df_pca['subject'].isin(TEST_SUBJECTS)].copy()
df_test = extract_lmad_feature(df_test, 'PCA_Mag', best_n)
df_test = extract_lmad_feature(df_test, 'PCA_Phase', best_n)

best_mag_col = f'PCA_Mag_ord{best_n}'
best_phase_col = f'PCA_Phase_ord{best_n}'
session_agg_test = segment_aggregate(df_test, [best_mag_col, best_phase_col])

test_results = []
for env in envs:
    env_data = session_agg_test[session_agg_test['environment'] == env]
    if env_data.empty:
        continue
    for modality in ['PCA_Mag', 'PCA_Phase']:
        feat_col = f'{modality}_ord{best_n}'
        valid = env_data[feat_col].notna()
        env_valid = env_data[valid]
        y_true = (env_valid['activity'] == 'walking').astype(int)
        X_val = env_valid[feat_col]
        if len(X_val) < 2 or y_true.nunique() < 2:
            continue
        auc_val = roc_auc_score(y_true, X_val)
        auc_val = max(auc_val, 1 - auc_val)
        print(f"   TEST | Env: {env:<8} | {modality:<9} | AUC: {auc_val:.4f} | n_seg={len(X_val)}")
        test_results.append({'Environment': env.capitalize(), 'Modality': modality, 'AUC': auc_val})

1. Loading & Filtering Data...
Length of df_raw: 143214
Length of df_clean: 137409
We have all the sessions
10 sessions with the least rows
   environment       session_id  row_count
49        foil  20251016_183114         48
48        foil  20251016_182927        830
1         foil  20251015_165100        895
0         foil  20251015_164911        897
2         foil  20251015_165246        899
3         foil  20251015_165542        903
82      nofoil  20251010_143219        917
27        foil  20251016_173831        918
74      nofoil  20251010_141559        918
81      nofoil  20251010_143031        918

Number of broken sessions (lengths < 100): 1
['walking']

2. Evaluating n_diff (1 to 8) using LOSO-CV on Training Subjects...
   (Segment size = 53 samples per evaluation unit)

   -> Order n=1
      Fold=matthew  | Env: nofoil   | PCA_Mag   | AUC: 0.9831 | Margin: 3.1384 | n_seg=180
      Fold=matthew  | Env: nofoil   | PCA_Phase | AUC: 0.9990 | Margin: 3.5368 | n_seg=180
      Fold

In [ ]:
# ============================================================================
# 6. GRAPH 1: Per-Fold Results (see each subject's contribution)
# ============================================================================
df_plot = pd.DataFrame(margin_data)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey='row')
fig.suptitle("Per-Fold (Held-Out Subject) Results by Environment", fontweight='bold', fontsize=14, y=1.02)

for col_idx, env in enumerate(['Nofoil', 'Foil', 'Open']):
    env_data = df_plot[df_plot['Environment'] == env]
    
    # Top row: AUC per fold
    sns.lineplot(
        data=env_data, x='n_diff', y='AUC', hue='Fold', style='Modality',
        markers=True, dashes=True, linewidth=1.5, markersize=7, ax=axes[0, col_idx]
    )
    axes[0, col_idx].set_title(f"{env} - AUC", fontweight='bold')
    axes[0, col_idx].set_xlabel("Differencing Order ($n$)")
    axes[0, col_idx].set_xticks(range(1, MAX_N_DIFF + 1))
    if col_idx > 0:
        axes[0, col_idx].get_legend().remove()
    else:
        axes[0, col_idx].legend(fontsize=8, loc='lower left')
    
    # Bottom row: Margin per fold
    sns.lineplot(
        data=env_data, x='n_diff', y='Margin', hue='Fold', style='Modality',
        markers=True, dashes=True, linewidth=1.5, markersize=7, ax=axes[1, col_idx]
    )
    axes[1, col_idx].set_title(f"{env} - Margin", fontweight='bold')
    axes[1, col_idx].set_xlabel("Differencing Order ($n$)")
    axes[1, col_idx].set_xticks(range(1, MAX_N_DIFF + 1))
    if col_idx > 0:
        axes[1, col_idx].get_legend().remove()
    else:
        axes[1, col_idx].legend(fontsize=8, loc='upper left')

axes[0, 0].set_ylabel("AUC", fontweight='bold')
axes[1, 0].set_ylabel("Separation Margin", fontweight='bold')

plt.tight_layout()
plt.savefig("1_Per_Fold_Results.png", dpi=300, bbox_inches='tight')
plt.show()
print("[SAVED] 1_Per_Fold_Results.png")

In [ ]:
# ============================================================================
# 7. GRAPH 2: Aggregated Mean +/- Std across folds (combined overview)
# ============================================================================
df_plot_stats = df_plot.groupby(['n_diff', 'Environment', 'Modality']).agg(
    AUC_mean=('AUC', 'mean'), AUC_std=('AUC', 'std'),
    Margin_mean=('Margin', 'mean'), Margin_std=('Margin', 'std')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"LOSO-CV Aggregated Results (Mean +/- Std across folds)", fontweight='bold', fontsize=13)

# Left: AUC with error bands
for (env, mod), grp in df_plot_stats.groupby(['Environment', 'Modality']):
    linestyle = '-' if mod == 'PCA_Mag' else '--'
    label = f"{env} / {'Mag' if 'Mag' in mod else 'Phase'}"
    axes[0].plot(grp['n_diff'], grp['AUC_mean'], marker='o', linestyle=linestyle, label=label, linewidth=1.5)
    axes[0].fill_between(grp['n_diff'],
                         grp['AUC_mean'] - grp['AUC_std'],
                         grp['AUC_mean'] + grp['AUC_std'], alpha=0.15)

axes[0].set_title("AUC (Mean +/- Std)", fontweight='bold')
axes[0].set_xlabel("Differencing Order ($n$)", fontweight='bold')
axes[0].set_ylabel("AUC", fontweight='bold')
axes[0].set_xticks(range(1, MAX_N_DIFF + 1))
axes[0].legend(fontsize=8, ncol=2)

# Right: Margin with error bands
for (env, mod), grp in df_plot_stats.groupby(['Environment', 'Modality']):
    linestyle = '-' if mod == 'PCA_Mag' else '--'
    label = f"{env} / {'Mag' if 'Mag' in mod else 'Phase'}"
    axes[1].plot(grp['n_diff'], grp['Margin_mean'], marker='s', linestyle=linestyle, label=label, linewidth=1.5)
    axes[1].fill_between(grp['n_diff'],
                         grp['Margin_mean'] - grp['Margin_std'],
                         grp['Margin_mean'] + grp['Margin_std'], alpha=0.15)

axes[1].set_title("Separation Margin (Mean +/- Std)", fontweight='bold')
axes[1].set_xlabel("Differencing Order ($n$)", fontweight='bold')
axes[1].set_ylabel("Margin", fontweight='bold')
axes[1].set_xticks(range(1, MAX_N_DIFF + 1))
axes[1].legend(fontsize=8, ncol=2)

# Mark the best n
for ax in axes:
    ax.axvline(x=best_n, color='red', linestyle=':', alpha=0.6, label=f'Best n={best_n}')

plt.tight_layout()
plt.savefig("2_Aggregated_Mean_Std.png", dpi=300, bbox_inches='tight')
plt.show()
print("[SAVED] 2_Aggregated_Mean_Std.png")

In [ ]:
# ============================================================================
# 8. GRAPH 3: Heatmap summary (all results at a glance)
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Summary Heatmaps: AUC and Margin (Mean across LOSO folds)", fontweight='bold', fontsize=13)

# Pivot for heatmap: rows = Env+Modality, cols = n_diff
df_plot_stats['Label'] = df_plot_stats['Environment'] + ' / ' + df_plot_stats['Modality'].str.replace('PCA_', '')

# AUC heatmap
auc_pivot = df_plot_stats.pivot(index='Label', columns='n_diff', values='AUC_mean')
sns.heatmap(auc_pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.5, vmax=1.0,
            linewidths=0.5, ax=axes[0], cbar_kws={'label': 'AUC'})
axes[0].set_title("Mean AUC", fontweight='bold')
axes[0].set_xlabel("Differencing Order ($n$)")
axes[0].set_ylabel("")

# Margin heatmap
margin_pivot = df_plot_stats.pivot(index='Label', columns='n_diff', values='Margin_mean')
sns.heatmap(margin_pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=axes[1], cbar_kws={'label': 'Margin'})
axes[1].set_title("Mean Separation Margin", fontweight='bold')
axes[1].set_xlabel("Differencing Order ($n$)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("3_Heatmap_Summary.png", dpi=300, bbox_inches='tight')
plt.show()
print("[SAVED] 3_Heatmap_Summary.png")

# ============================================================================
# 9. Summary Table
# ============================================================================
print("\n" + "="*70)
print(f"SELECTED: n_diff = {best_n}")
print(f"  LOSO-CV Mean AUC:    {best_avg_auc:.4f}")
print(f"  LOSO-CV Mean Margin: {best_avg_margin:.4f}")
print("="*70)
if test_results:
    print(f"\nFinal TEST set results (n={best_n}):")
    for r in test_results:
        print(f"  {r['Environment']:<8} | {r['Modality']:<9} | AUC: {r['AUC']:.4f}")
